In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input # Thêm Input vào đây
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# 1. TIỀN XỬ LÝ DỮ LIỆU
# Thiết lập đường dẫn dữ liệu
train_dir = ""

# Kích thước ảnh và kích thước lô
img_width, img_height = 128, 128
batch_size = 32

# Tăng cường dữ liệu dành cho huấn luyện mô hình
train_datagen = ImageDataGenerator(
    rescale=1.0/255, # Normalize pixel values
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2 # Tách 20% dữ liệu cho tập xác thực
)

In [ ]:
" tải dữ liệu huấn luyện"

In [ ]:
# Tải dữ liệu huấn luyện
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset='training' # Chỉ định đây là tập huấn luyện
)

# Tải dữ liệu xác thực
validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset='validation' # Chỉ định đây là tập xác thực
)

In [ ]:
"xây dựng mô hình"

In [ ]:

# XÂY DỰNG MÔ HÌNH CNN
model = Sequential([
    Input(shape=(img_width, img_height, 3)), # Sử dụng lớp Input rõ ràng
    Conv2D(32, (3,3), activation="relu"), # Loại bỏ input_shape ở đây
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5), # Reduce overfitting
    Dense(38, activation="softmax") 
])

In [ ]:
# Biên dịch mô hình
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

# Tóm tắt cấu hình của mô hình
model.summary()

# HUẤN LUYỆN MÔ HÌNH CNN
epochs = 50
history = model.fit(train_generator, epochs=epochs, validation_data=validation_generator) # Thêm validation_data

# ĐÁNH GIÁ KẾT QUẢ MÔ HÌNH
plt.plot(history.history['accuracy'], label="Kết quả huấn luyện")
plt.plot(history.history['val_accuracy'], label="Độ chính xác xác thực") # Bỏ comment dòng này
plt.xlabel("Số lần học")
plt.ylabel("Độ chính xác")
plt.legend()
plt.show()

In [ ]:
model.save('nhận diện khuôn mặt.h5')

In [ ]:
from google.colab import files
files.download('nhận diện khuôn mặt.h5')

In [ ]:
from keras.utils import load_img
import numpy as np
from keras.models import load_model # Import load_model

# Load the saved model
saved_model = load_model('nhận diện khuôn mặt.h5') # Load your saved model

path = ""

# Tải và tiền xử lý ảnh kiểm tra
img = load_img(path, target_size=(128, 128))
plt.imshow(img)
plt.show()
img = np.array(img)
img = img / 255.0
img = img.reshape(1, 128, 128, 3)

# Tiên đoán loại
prediction = np.argmax(saved_model.predict(img)) # Use the loaded model for prediction

class_labels = {v: k for k, v in train_generator.class_indices.items()}
person_name = class_labels[prediction]
print(f"Người tiên đoán: {person_name}")